# M0 — Build final classification dataset from 4 SIMPLE source datasets — FIX2

Notebook này tạo **bộ DATA cuối cùng cho bài toán image classification 6 mệnh giá** sau Visual Audit V2.

FIX2 sửa đúng hai điểm của bản trước:

1. **Split leakage-aware thực sự**: không còn coi gần như mỗi ảnh là một group. `money_v1` được giữ theo ngày chụp; `banknotes_v2` dùng capture lineage đã có từ notebook M0 cũ; exact SHA/UUID lineage tiếp tục được nối xuyên nguồn khi có evidence.
2. **Tên ảnh final thống nhất** theo class, không còn prefix nguồn:
   - `10000_000001.png`, `10000_000002.png`, ...
   - `20000_000001.png`, ...
   - tương tự cho 6 class.

Chỉ filename được làm sạch; toàn bộ provenance gốc vẫn nằm trong `manifest.csv`.

Luồng:

1. Load 4 output SIMPLE mới nhất.
2. Áp dụng Visual Audit V2: 9.984 → loại 23 → giữ 9.961.
3. Không balance.
4. Tạo real leakage groups.
5. Group-aware split khoảng 80/10/10.
6. Gán filename canonical theo từng class.
7. Dựng `C:\CS231\DATA_NEW_M0_FIX2`.
8. Audit integrity + leakage + naming.
9. Chỉ sau PASS mới thay `C:\CS231\DATA`.

Không train model. Không commit/push GitHub.


## 1. Imports và cấu hình

Quyết định canonical hiện tại:

- final retained pool = **9.961 ảnh** sau Classification-Centric Visual Audit V2;
- **không balance / không undersample**;
- split mục tiêu khoảng **80% train / 10% dev / 10% test**;
- group/session leakage safety quan trọng hơn ép tỷ lệ tuyệt đối;
- filename final = `<denomination>_<6-digit-index>.png`, đánh số **toàn class trước split**, nên tên ảnh ổn định dù split thay đổi.


In [1]:
from pathlib import Path
from datetime import datetime
from PIL import Image
from IPython.display import display
import pandas as pd
import hashlib
import shutil
import re

PROJECT_ROOT = Path(r"C:\CS231")
M0_DIR = PROJECT_ROOT / "src" / "M0"
RUNS_ROOT = Path(r"C:\CS231_RAW_NOTEBOOK_RUNS")

TARGET_CLASSES = ["10000", "20000", "50000", "100000", "200000", "500000"]
DECISIONS_CSV = M0_DIR / "COMBINED_VISUAL_REVIEW_DECISIONS_CHATGPT_CLASSIFICATION_AUDIT_V2.csv"

SOURCE_ROOTS = {
    "banknotes_v2": RUNS_ROOT / "banknotes_v2_simple",
    "money_v1": RUNS_ROOT / "money_v1_simple",
    "blindtrix_v1": RUNS_ROOT / "blindtrix_v1_simple",
    "currency_v6": RUNS_ROOT / "currency_v6_simple",
}

STAGING_ROOT = PROJECT_ROOT / "DATA_NEW_M0_FIX2"
FINAL_DATA_ROOT = PROJECT_ROOT / "DATA"

TRAIN_RATIO = 0.80
DEV_RATIO = 0.10
TEST_RATIO = 0.10
SPLIT_SEED = 2312026

assert abs(TRAIN_RATIO + DEV_RATIO + TEST_RATIO - 1.0) < 1e-9
assert PROJECT_ROOT.is_dir(), f"Không tìm thấy project: {PROJECT_ROOT}"
assert M0_DIR.is_dir(), f"Không tìm thấy M0: {M0_DIR}"
assert DECISIONS_CSV.is_file(), (
    f"Thiếu file Visual Audit V2:\n{DECISIONS_CSV}\n\n"
    "Hãy đặt CSV audit V2 cạnh Notebook 05 trong src\\M0 rồi chạy lại."
)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DECISIONS_CSV:", DECISIONS_CSV)
print("STAGING_ROOT FIX2:", STAGING_ROOT)
print("FINAL_DATA_ROOT hiện tại:", FINAL_DATA_ROOT)


PROJECT_ROOT: C:\CS231
DECISIONS_CSV: C:\CS231\src\M0\COMBINED_VISUAL_REVIEW_DECISIONS_CHATGPT_CLASSIFICATION_AUDIT_V2.csv
STAGING_ROOT FIX2: C:\CS231\DATA_NEW_M0_FIX2
FINAL_DATA_ROOT hiện tại: C:\CS231\DATA


## 2. Helper: tìm manifest nguồn mới nhất và chuẩn hóa schema

Notebook **không dùng DATA cũ làm nguồn**. Nó chỉ đọc output của 4 notebook SIMPLE trong `C:\CS231_RAW_NOTEBOOK_RUNS`.

Do các manifest SIMPLE được tạo ở các thời điểm khác nhau, tên cột class có thể là `final_label`, `denomination` hoặc `raw_label`. Notebook 05 chuẩn hóa chúng về **một schema canonical** trước khi merge, nhưng **không thay đổi sample selection**.

Ưu tiên nhãn:

`final_label → denomination → raw_label`

Các lineage key quan trọng như `source_name` không được tự suy diễn nếu thật sự bị thiếu.


In [2]:
def latest_manifest(source_root: Path) -> Path:
    assert source_root.is_dir(), f"Không tìm thấy source run root: {source_root}"
    hits = [p for p in source_root.rglob("manifest.csv") if p.is_file()]
    assert hits, f"Không tìm thấy manifest.csv dưới {source_root}"
    hits.sort(key=lambda p: (p.stat().st_mtime_ns, str(p)), reverse=True)
    return hits[0]

def clean_text_series(series):
    return series.fillna("").astype(str)

def normalize_denomination_value(value):
    """
    Chuẩn hóa các cách ghi class phổ biến về đúng 6 denomination canonical.
    Ví dụ: 10000, 10000.0, '10 000 VND' -> '10000'.
    """
    s = str(value).strip()
    if not s:
        return ""

    if re.fullmatch(r"\d+(?:\.0+)?", s):
        try:
            s = str(int(float(s)))
        except Exception:
            pass

    if s in TARGET_CLASSES:
        return s

    digits = re.sub(r"\D", "", s)
    return digits if digits in TARGET_CLASSES else s

def adapt_source_manifest_schema(df, expected_dataset_id):
    """
    Bốn notebook SIMPLE có cùng pipeline nhưng manifest lịch sử có thể khác tên cột.
    Hàm này CHỈ chuẩn hóa schema; không thay đổi selection/data semantics.

    Class label priority:
      final_label -> denomination -> raw_label

    Lý do ưu tiên final_label:
      nếu source notebook đã áp dụng VERIFIED relabel thì final_label mới là nhãn
      cần đưa sang dataset classification cuối.
    """
    df = df.copy()
    notes = []

    # Path alias.
    if "relative_path" not in df.columns:
        for alias in ["crop_relative_path", "final_relative_path"]:
            if alias in df.columns:
                df["relative_path"] = df[alias]
                notes.append(f"relative_path <- {alias}")
                break

    # dataset_id có thể điền an toàn vì chính SOURCE_ROOTS xác định source.
    if "dataset_id" not in df.columns:
        df["dataset_id"] = expected_dataset_id
        notes.append("dataset_id <- expected source id")
    else:
        dataset_values = df["dataset_id"].fillna("").astype(str).str.strip()
        if (dataset_values == "").all():
            df["dataset_id"] = expected_dataset_id
            notes.append("dataset_id empty -> expected source id")

    # Không tự chế source_name vì đây là lineage key quan trọng.
    if "source_name" not in df.columns:
        for alias in ["canonical_source_name", "base_source_name"]:
            if alias in df.columns:
                df["source_name"] = df[alias]
                notes.append(f"source_name <- {alias}")
                break

    # Hỗ trợ một vài alias mô tả cùng bbox gate.
    if "bbox_short_side_px" not in df.columns:
        for alias in ["bbox_short_side", "bbox_min_side_px", "bbox_short_side_pixels"]:
            if alias in df.columns:
                df["bbox_short_side_px"] = df[alias]
                notes.append(f"bbox_short_side_px <- {alias}")
                break

    # Chuẩn hóa label về cột denomination canonical.
    label_col = None
    for candidate in ["final_label", "denomination", "raw_label"]:
        if candidate not in df.columns:
            continue

        normalized = df[candidate].map(normalize_denomination_value)
        nonempty = normalized[normalized != ""]

        if len(nonempty) and set(nonempty).issubset(TARGET_CLASSES):
            label_col = candidate
            df["denomination"] = normalized
            notes.append(f"denomination <- {candidate}")
            break

    assert label_col is not None, (
        f"{expected_dataset_id}: không tìm thấy cột class hợp lệ. "
        f"Các cột hiện có: {list(df.columns)}"
    )

    # Các cột thật sự bắt buộc cho final build/leakage audit.
    required = {
        "relative_path",
        "dataset_id",
        "source_name",
        "bbox_short_side_px",
    }
    missing = sorted(required - set(df.columns))
    assert not missing, (
        f"{expected_dataset_id}: sau schema adapter vẫn thiếu cột bắt buộc {missing}. "
        f"Các cột hiện có: {list(df.columns)}"
    )

    # Provenance bổ sung: nếu source notebook không có thì để trống, KHÔNG suy diễn.
    for col in [
        "source_archive_member",
        "annotation_member",
        "capture_group",
        "capture_date",
        "capture_time",
        "temporal_cluster_id",
        "source_image_sha256",
        "final_crop_sha256",
    ]:
        if col not in df.columns:
            df[col] = ""

    # Canonical text columns.
    for col in [
        "dataset_id",
        "source_name",
        "relative_path",
        "denomination",
        "source_archive_member",
        "annotation_member",
        "capture_group",
        "capture_date",
        "capture_time",
        "temporal_cluster_id",
        "source_image_sha256",
        "final_crop_sha256",
    ]:
        df[col] = clean_text_series(df[col])

    return df, label_col, notes

manifest_paths = {}
source_frames = []
schema_audit_rows = []

for expected_dataset_id, root in SOURCE_ROOTS.items():
    manifest_path = latest_manifest(root)
    raw_df = pd.read_csv(manifest_path, keep_default_na=False)

    df, label_col, schema_notes = adapt_source_manifest_schema(
        raw_df, expected_dataset_id
    )

    unique_dataset_ids = sorted(
        x for x in df["dataset_id"].astype(str).str.strip().unique() if x
    )
    assert unique_dataset_ids == [expected_dataset_id], (
        f"Dataset id mismatch: expected {expected_dataset_id}, "
        f"got {unique_dataset_ids}"
    )

    assert set(df["denomination"]).issubset(TARGET_CLASSES), (
        f"{expected_dataset_id}: có label ngoài 6 class: "
        f"{sorted(set(df['denomination']) - set(TARGET_CLASSES))}"
    )

    # Gate 120 px phải vẫn đúng với source output final.
    bbox_numeric = pd.to_numeric(df["bbox_short_side_px"], errors="coerce")
    assert bbox_numeric.notna().all(), (
        f"{expected_dataset_id}: bbox_short_side_px có giá trị không phải số"
    )
    assert (bbox_numeric >= 120).all(), (
        f"{expected_dataset_id}: phát hiện sample vi phạm bbox >= 120 px"
    )
    df["bbox_short_side_px"] = bbox_numeric.astype(int)

    assert df["source_name"].astype(str).str.strip().ne("").all(), (
        f"{expected_dataset_id}: source_name trống; không thể audit lineage an toàn"
    )
    assert df["source_name"].duplicated().sum() == 0, (
        f"{expected_dataset_id}: source_name bị trùng trong final source manifest"
    )
    assert df["relative_path"].duplicated().sum() == 0, (
        f"{expected_dataset_id}: relative_path bị trùng trong final source manifest"
    )

    df["source_manifest_path"] = str(manifest_path)
    df["source_dataset_root"] = str(manifest_path.parent)

    manifest_paths[expected_dataset_id] = manifest_path
    source_frames.append(df)

    schema_audit_rows.append({
        "dataset_id": expected_dataset_id,
        "rows": len(df),
        "label_source_column": label_col,
        "schema_notes": " | ".join(schema_notes) if schema_notes else "canonical",
        "manifest_path": str(manifest_path),
    })

    print(
        f"{expected_dataset_id}: {len(df):,} rows | "
        f"label <- {label_col} | {manifest_path}"
    )

source_all = pd.concat(source_frames, ignore_index=True)

print("\nSchema adapter audit:")
display(pd.DataFrame(schema_audit_rows))

print("\nTổng source candidates:", f"{len(source_all):,}")
display(
    source_all.groupby(["dataset_id", "denomination"])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=TARGET_CLASSES, fill_value=0)
)

# Đây là regression gate của lần build hiện tại:
# 1329 + 6378 + 1998 + 279 = 9984 candidates trước Visual Audit V2.
assert len(source_all) == 9984, (
    f"Tổng 4 source output phải là 9984 candidates, thực tế = {len(source_all)}. "
    "Dừng để audit source runs trước khi merge."
)


banknotes_v2: 1,329 rows | label <- final_label | C:\CS231_RAW_NOTEBOOK_RUNS\banknotes_v2_simple\20261026_003633\BANKNOTES_CLEAN\manifest.csv
money_v1: 6,378 rows | label <- final_label | C:\CS231_RAW_NOTEBOOK_RUNS\money_v1_simple\20261026_004048\MONEY_V1_CLEAN\manifest.csv
blindtrix_v1: 1,998 rows | label <- denomination | C:\CS231_RAW_NOTEBOOK_RUNS\blindtrix_v1_simple\20261026_002923_926824\BLINDTRIX_V1_CLEAN\manifest.csv
currency_v6: 279 rows | label <- denomination | C:\CS231_RAW_NOTEBOOK_RUNS\currency_v6_simple\20261026_003443_922635\CURRENCY_V6_CLEAN\manifest.csv

Schema adapter audit:


,dataset_id,rows,label_source_column,schema_notes,manifest_path
0,banknotes_v2,1329,final_label,denomination <- final_label,C:\CS231_RAW_NOTEBOOK_RUNS\banknotes_v2_simple...
1,money_v1,6378,final_label,denomination <- final_label,C:\CS231_RAW_NOTEBOOK_RUNS\money_v1_simple\202...
2,blindtrix_v1,1998,denomination,denomination <- denomination,C:\CS231_RAW_NOTEBOOK_RUNS\blindtrix_v1_simple...
3,currency_v6,279,denomination,denomination <- denomination,C:\CS231_RAW_NOTEBOOK_RUNS\currency_v6_simple\...



Tổng source candidates: 9,984


denomination,10000,20000,50000,100000,200000,500000
dataset_id,,,,,,
banknotes_v2,285,147,159,308,217,213
blindtrix_v1,354,359,293,276,357,359
currency_v6,59,58,14,10,66,72
money_v1,1038,912,883,1077,1519,949


## 3. Áp dụng Visual Audit V2 — classification-centric

Visual Audit V2 đã xem lại **100% 9.984 candidate crops** theo đúng bài toán classification.

Quyết định đã chốt:

- `KEEP`: 9.961
- `REMOVE`: 23
- `RELABEL`: 0

Notebook kiểm tra lại các con số này trước khi làm bất cứ bước downstream nào.


In [3]:
decisions = pd.read_csv(DECISIONS_CSV, keep_default_na=False, dtype=str)

required_decision_cols = {
    "visual_review_id", "dataset_id", "source_name",
    "old_label", "decision", "new_label", "reason"
}
missing = sorted(required_decision_cols - set(decisions.columns))
assert not missing, f"Decision CSV thiếu cột: {missing}"

for col in required_decision_cols:
    decisions[col] = clean_text_series(decisions[col])

assert len(decisions) == 9984, f"Expected 9984 decisions, got {len(decisions)}"
assert decisions["visual_review_id"].nunique() == 9984
assert decisions[["dataset_id", "source_name"]].duplicated().sum() == 0
assert set(decisions["decision"]) <= {"KEEP", "REMOVE", "RELABEL"}

decision_counts = decisions["decision"].value_counts().to_dict()
assert decision_counts.get("KEEP", 0) == 9961, decision_counts
assert decision_counts.get("REMOVE", 0) == 23, decision_counts
assert decision_counts.get("RELABEL", 0) == 0, decision_counts

merged = source_all.merge(
    decisions[[
        "visual_review_id", "dataset_id", "source_name",
        "old_label", "decision", "new_label", "reason"
    ]],
    on=["dataset_id", "source_name"],
    how="outer",
    validate="one_to_one",
    indicator=True
)

assert len(merged) == 9984
assert (merged["_merge"] == "both").all(), (
    "Manifest 4 nguồn và Visual Audit V2 không khớp 1-1. Dừng tại đây."
)
merged = merged.drop(columns="_merge")

assert (merged["denomination"].astype(str) == merged["old_label"].astype(str)).all(), (
    "Có mismatch giữa label trong source manifest và label đã visual-review."
)

merged["final_label"] = merged["denomination"].astype(str)
relabel_mask = merged["decision"] == "RELABEL"
if relabel_mask.any():
    assert merged.loc[relabel_mask, "new_label"].isin(TARGET_CLASSES).all()
    merged.loc[relabel_mask, "final_label"] = merged.loc[relabel_mask, "new_label"]

removed = merged[merged["decision"] == "REMOVE"].copy()
kept = merged[merged["decision"] != "REMOVE"].copy().reset_index(drop=True)

assert len(removed) == 23
assert len(kept) == 9961
assert set(kept["final_label"]).issubset(TARGET_CLASSES)

print("Visual Audit V2:")
print("  candidates =", len(merged))
print("  retained =", len(kept))
print("  REMOVE =", len(removed))
print("\nClass counts sau Visual Audit V2 — KHÔNG BALANCE:")
display(
    kept.groupby("final_label")
    .size()
    .reindex(TARGET_CLASSES, fill_value=0)
    .to_frame("images")
)


Visual Audit V2:
  candidates = 9984
  retained = 9961
  REMOVE = 23

Class counts sau Visual Audit V2 — KHÔNG BALANCE:


,images
final_label,
10000,1730
20000,1475
50000,1348
100000,1671
200000,2144
500000,1593


## 4. Tạo leakage group thật — FIX2

Bản trước bị HOLD vì `9.961 ảnh = 9.961 group`, tức gần như image-level split.

FIX2 dùng provenance thật theo từng nguồn:

- **money_v1**: mọi ảnh cùng `capture_date` ở cùng một group. Đây là quy tắc bảo thủ, dễ giải thích và ngăn cùng ngày chụp rơi sang nhiều split. Nếu thiếu ngày thì fallback về `temporal_cluster_id`, rồi mới tới `source_name`.
- **banknotes_v2**: tái sử dụng capture-lineage từ notebook M0 cũ:
  - chuỗi `DENOMINATION_frame` → block 10 frame;
  - `IMG_number` → block 50 frame;
  - timestamp-like names → cùng bucket lịch sử;
  - không match thì fallback `source_name`.
- **blindtrix_v1 / currency_v6**: public source không có session metadata đáng tin cậy, nên fallback `source_name`.
- Sau đó union thêm **exact source SHA**, **exact final-crop SHA** và UUID-like source stem xuyên dataset.

Ta không claim physical-banknote-disjoint vì public source không cung cấp physical-note ID đáng tin cậy.


In [4]:
def nonempty(value):
    return str(value).strip()


def uuid_like(text):
    text = nonempty(text)
    return bool(re.fullmatch(
        r"[0-9a-fA-F]{8}-[0-9a-fA-F]{4}-[0-9a-fA-F]{4}-[0-9a-fA-F]{4}-[0-9a-fA-F]{12}",
        text,
    ))


def banknotes_capture_key(row):
    """Tái sử dụng logic capture lineage đã có trong Notebook 01 lịch sử."""
    name = nonempty(row.get("source_name", ""))

    timestamp = re.match(r"(\d{6,8})_(\d{4,6})_", name)
    if timestamp:
        date, time = timestamp.groups()
        hour_prefix = int(time[:2]) if len(time) >= 2 else 0
        return (
            f"banknotes_v2::timestamp::{date}::hblock{hour_prefix // 10}",
            "banknotes_historical_timestamp_bucket",
        )

    sequence = re.match(r"(\d{6})_(\d+)$", name)
    if sequence:
        denomination, frame = sequence.groups()
        return (
            f"banknotes_v2::seq::{denomination}::block{int(frame) // 10:04d}",
            "banknotes_historical_sequence_block10",
        )

    camera = re.match(r"IMG_(\d+)", name, flags=re.I)
    if camera:
        return (
            f"banknotes_v2::IMG::block{int(camera.group(1)) // 50:05d}",
            "banknotes_historical_IMG_block50",
        )

    return f"banknotes_v2::source::{name}", "source_name_fallback"


def money_capture_key(row):
    """Ưu tiên ngày chụp để cùng ngày không bao giờ rơi sang nhiều split."""
    capture_date = nonempty(row.get("capture_date", ""))
    if capture_date:
        return f"money_v1::date::{capture_date}", "money_capture_date"

    # Fallback: parse trực tiếp IMG_YYYYMMDD_HHMMSS nếu manifest chưa giữ capture_date.
    search_text = " | ".join([
        nonempty(row.get("source_name", "")),
        nonempty(row.get("source_archive_member", "")),
    ])
    match = re.search(r"IMG_(\d{8})_(\d{6})", search_text, flags=re.I)
    if match:
        return f"money_v1::date::{match.group(1)}", "money_date_parsed_from_filename"

    temporal = nonempty(row.get("temporal_cluster_id", ""))
    if temporal:
        return temporal, "money_temporal_cluster_fallback"

    return (
        f"money_v1::source::{nonempty(row.get('source_name', ''))}",
        "source_name_fallback",
    )


def base_group_for_row(row):
    dataset_id = nonempty(row.get("dataset_id", ""))
    if dataset_id == "banknotes_v2":
        return banknotes_capture_key(row)
    if dataset_id == "money_v1":
        return money_capture_key(row)

    source = nonempty(row.get("source_name", ""))
    return f"{dataset_id}::source::{source}", "source_name_fallback"


# Base group + evidence method.
base_keys = []
group_methods = []
for row in kept.to_dict("records"):
    key, method = base_group_for_row(row)
    base_keys.append(key)
    group_methods.append(method)

kept["base_group_key"] = base_keys
kept["group_method"] = group_methods

# Union-find trên retained rows.
n = len(kept)
parent = list(range(n))
rank = [0] * n


def find(i):
    while parent[i] != i:
        parent[i] = parent[parent[i]]
        i = parent[i]
    return i


def union(a, b):
    ra, rb = find(a), find(b)
    if ra == rb:
        return
    if rank[ra] < rank[rb]:
        ra, rb = rb, ra
    parent[rb] = ra
    if rank[ra] == rank[rb]:
        rank[ra] += 1


def union_by_token(tokens):
    first = {}
    for i, token in enumerate(tokens):
        token = nonempty(token)
        if not token:
            continue
        if token in first:
            union(i, first[token])
        else:
            first[token] = i


# 1) Session / capture lineage.
union_by_token(kept["base_group_key"].tolist())

# 2) Exact byte identity across sources when available.
union_by_token(kept["source_image_sha256"].tolist())
union_by_token(kept["final_crop_sha256"].tolist())

# 3) UUID-like lineage xuyên blindtrix/currency khi cùng public source stem.
uuid_tokens = [
    f"UUID::{name.lower()}" if uuid_like(name) else ""
    for name in kept["source_name"].tolist()
]
union_by_token(uuid_tokens)

# Deterministic group IDs ordered by smallest visual-review ID in each component.
components = {}
for i in range(n):
    root = find(i)
    components.setdefault(root, []).append(i)

component_order = sorted(
    components,
    key=lambda root: min(
        int(re.sub(r"\D", "", kept.iloc[i]["visual_review_id"]))
        for i in components[root]
    ),
)
root_to_group = {
    root: f"G{group_number:05d}"
    for group_number, root in enumerate(component_order, start=1)
}
kept["final_group"] = [root_to_group[find(i)] for i in range(n)]

group_sizes = kept.groupby("final_group").size()

# Group audit theo source.
source_group_audit = []
for dataset_id, frame in kept.groupby("dataset_id"):
    sizes = frame.groupby("final_group").size()
    source_group_audit.append({
        "dataset_id": dataset_id,
        "images": len(frame),
        "groups_touching_source": frame["final_group"].nunique(),
        "singleton_groups_within_source_view": int((sizes == 1).sum()),
        "max_group_size_within_source_view": int(sizes.max()),
        "group_methods": ", ".join(sorted(frame["group_method"].unique())),
    })

print("Leakage-group audit FIX2:")
display(pd.DataFrame(source_group_audit))

print("Retained images:", len(kept))
print("Final leakage groups:", kept["final_group"].nunique())
print("Largest group size:", int(group_sizes.max()))
print("Non-singleton groups:", int((group_sizes > 1).sum()))

# FIX2 phải thật sự tạo được group nhiều ảnh cho các source có session lineage.
assert kept["final_group"].nunique() < len(kept), (
    "FIX2 vẫn tạo 1 group/ảnh. Dừng: session grouping chưa hoạt động."
)
assert int(group_sizes.max()) > 1

money = kept[kept["dataset_id"] == "money_v1"]
banknotes = kept[kept["dataset_id"] == "banknotes_v2"]
assert money["final_group"].nunique() < len(money), (
    "money_v1 vẫn singleton toàn bộ; capture-date grouping chưa hoạt động."
)
assert banknotes["final_group"].nunique() < len(banknotes), (
    "banknotes_v2 vẫn singleton toàn bộ; capture-lineage grouping chưa hoạt động."
)

# Audit rất quan trọng: cùng capture_date money_v1 không được map sang nhiều final_group.
money_dates = money[money["capture_date"].astype(str).str.strip() != ""]
if len(money_dates):
    date_group_n = money_dates.groupby("capture_date")["final_group"].nunique()
    assert int((date_group_n > 1).sum()) == 0, (
        "Có capture_date của money_v1 bị tách thành nhiều group."
    )
    print("money_v1 capture dates:", money_dates["capture_date"].nunique())
else:
    print("WARN: money_v1 không có capture_date trong manifest; đã dùng fallback lineage.")


Leakage-group audit FIX2:


,dataset_id,images,groups_touching_source,singleton_groups_within_source_view,max_group_size_within_source_view,group_methods
0,banknotes_v2,1329,144,1,127,"banknotes_historical_IMG_block50, banknotes_hi..."
1,blindtrix_v1,1998,1998,1998,1,source_name_fallback
2,currency_v6,279,279,279,1,source_name_fallback
3,money_v1,6355,41,10,1718,"money_capture_date, source_name_fallback"


Retained images: 9961
Final leakage groups: 2446
Largest group size: 1718
Non-singleton groups: 190
money_v1 capture dates: 32


## 5. Group-aware split khoảng 80/10/10 — FIX2

Group là **đơn vị không thể tách**.

Vì một group có thể chứa nhiều denomination (ví dụ cùng ngày chụp `money_v1`), ta không stratify từng ảnh nữa. Thay vào đó:

1. tính vector số ảnh 6 class của từng group;
2. xử lý group lớn trước;
3. đưa mỗi group vào split đang **cần class mix đó nhất** để tiến gần 80/10/10;
4. tuyệt đối không bẻ group chỉ để đẹp tỷ lệ.

Kết quả phải audit:
- một `final_group` chỉ xuất hiện trong đúng một split;
- cùng `capture_date` của money_v1 không xuất hiện xuyên split;
- mọi class có mặt ở train/dev/test.


In [5]:
def stable_hash(text):
    payload = f"{SPLIT_SEED}|{text}".encode("utf-8")
    return hashlib.sha256(payload).hexdigest()


# Group x class count matrix.
group_class_counts = (
    kept.groupby(["final_group", "final_label"])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=TARGET_CLASSES, fill_value=0)
)
group_totals = group_class_counts.sum(axis=1)

class_totals = kept.groupby("final_label").size().reindex(TARGET_CLASSES, fill_value=0)
ratios = {"train": TRAIN_RATIO, "dev": DEV_RATIO, "test": TEST_RATIO}
split_order = ["train", "dev", "test"]

target_class = {
    split: {label: float(class_totals[label]) * ratio for label in TARGET_CLASSES}
    for split, ratio in ratios.items()
}
target_total = {split: len(kept) * ratio for split, ratio in ratios.items()}

current_class = {
    split: {label: 0 for label in TARGET_CLASSES}
    for split in split_order
}
current_total = {split: 0 for split in split_order}

# Group lớn trước; hash chỉ dùng deterministic tie-break.
group_order = sorted(
    group_class_counts.index,
    key=lambda g: (-int(group_totals[g]), stable_hash(g)),
)

assignments = {}

for group_id in group_order:
    gvec = {label: int(group_class_counts.loc[group_id, label]) for label in TARGET_CLASSES}
    gtotal = int(group_totals[group_id])

    candidates = []
    for split in split_order:
        # Gain = số sample của group đang lấp đúng phần target còn thiếu.
        class_gain = 0.0
        class_overflow = 0.0
        for label in TARGET_CLASSES:
            remaining = max(target_class[split][label] - current_class[split][label], 0.0)
            class_gain += min(gvec[label], remaining)
            class_overflow += max(
                current_class[split][label] + gvec[label] - target_class[split][label],
                0.0,
            )

        total_remaining = max(target_total[split] - current_total[split], 0.0)
        total_gain = min(gtotal, total_remaining)
        total_overflow = max(current_total[split] + gtotal - target_total[split], 0.0)

        # Ưu tiên lấp class need; phạt overflow. Không có random image-level split.
        score = (
            class_gain
            + 0.25 * total_gain
            - 2.0 * class_overflow
            - 1.0 * total_overflow
        )

        candidates.append((score, -split_order.index(split), split))

    _, _, chosen = max(candidates)
    assignments[group_id] = chosen
    current_total[chosen] += gtotal
    for label in TARGET_CLASSES:
        current_class[chosen][label] += gvec[label]

kept["split"] = kept["final_group"].map(assignments)
assert kept["split"].notna().all()
assert set(kept["split"]) == {"train", "dev", "test"}

# Core leakage gate.
group_split_counts = kept.groupby("final_group")["split"].nunique()
assert int((group_split_counts > 1).sum()) == 0

# money_v1: cùng ngày chụp không được xuyên split.
money = kept[kept["dataset_id"] == "money_v1"]
money_dates = money[money["capture_date"].astype(str).str.strip() != ""]
money_date_overlap = 0
if len(money_dates):
    money_date_overlap = int(
        (money_dates.groupby("capture_date")["split"].nunique() > 1).sum()
    )
assert money_date_overlap == 0

# banknotes_v2: cùng base capture group không được xuyên split.
banknotes = kept[kept["dataset_id"] == "banknotes_v2"]
banknotes_capture_overlap = int(
    (banknotes.groupby("base_group_key")["split"].nunique() > 1).sum()
)
assert banknotes_capture_overlap == 0

split_table = (
    kept.groupby(["split", "final_label"])
    .size()
    .unstack(fill_value=0)
    .reindex(index=split_order, columns=TARGET_CLASSES, fill_value=0)
)
split_table["TOTAL"] = split_table.sum(axis=1)

print("Split counts FIX2:")
display(split_table)
print("\nSplit percentages:")
display((split_table["TOTAL"] / len(kept) * 100).round(2).to_frame("percent"))

print("\nLeakage gates:")
print("  cross-split final_group overlap =", int((group_split_counts > 1).sum()))
print("  money_v1 capture_date overlap =", money_date_overlap)
print("  banknotes_v2 capture-group overlap =", banknotes_capture_overlap)

# Mỗi class phải có mặt ở cả 3 split.
assert (split_table[TARGET_CLASSES] > 0).all().all()

# Guard chống split bệnh lý; group integrity vẫn ưu tiên hơn exact ratio.
overall_pct = split_table["TOTAL"] / len(kept)
assert 0.70 <= overall_pct["train"] <= 0.90
assert 0.04 <= overall_pct["dev"] <= 0.20
assert 0.04 <= overall_pct["test"] <= 0.20


Split counts FIX2:


final_label,10000,20000,50000,100000,200000,500000,TOTAL
split,,,,,,,
train,1384,1180,1078,1337,1715,1275,7969
dev,173,147,135,167,215,159,996
test,173,148,135,167,214,159,996



Split percentages:


,percent
split,
train,80.0
dev,10.0
test,10.0



Leakage gates:
  cross-split final_group overlap = 0
  money_v1 capture_date overlap = 0
  banknotes_v2 capture-group overlap = 0


## 6. Gán filename canonical và dựng `DATA_NEW_M0_FIX2`

Tên final không còn chứa `BANKNOTES`, `MONEY`, `BLINDTRIX`, `CURRENCY`, source stem hay Visual Review ID.

Mỗi class được đánh số toàn cục, deterministic theo `visual_review_id`:

- `10000_000001.png` … `10000_001730.png`
- `20000_000001.png` … `20000_001475.png`
- `50000_000001.png` … `50000_001348.png`
- `100000_000001.png` … `100000_001671.png`
- `200000_000001.png` … `200000_002144.png`
- `500000_000001.png` … `500000_001593.png`

Số **không reset theo split**. Vì vậy filename là identity ổn định của dataset CS231 final. Provenance đầy đủ vẫn nằm trong manifest.


In [6]:
# Gán class index trước split để tên ảnh ổn định.
kept["visual_review_number"] = kept["visual_review_id"].str.extract(r"(\d+)", expand=False).astype(int)
kept["class_index"] = 0

for label in TARGET_CLASSES:
    ordered_idx = (
        kept[kept["final_label"] == label]
        .sort_values(["visual_review_number", "dataset_id", "source_name"])
        .index
        .tolist()
    )
    for class_index, row_index in enumerate(ordered_idx, start=1):
        kept.at[row_index, "class_index"] = class_index

kept["class_index"] = kept["class_index"].astype(int)
kept["final_filename"] = kept.apply(
    lambda row: f"{row['final_label']}_{int(row['class_index']):06d}.png",
    axis=1,
)

# Naming audit trước khi copy.
assert kept["final_filename"].nunique() == len(kept)
for label in TARGET_CLASSES:
    one = kept[kept["final_label"] == label].sort_values("class_index")
    expected = list(range(1, len(one) + 1))
    assert one["class_index"].tolist() == expected
    assert one["final_filename"].str.match(rf"^{label}_\d{{6}}\.png$").all()

print("Canonical filename ranges:")
for label in TARGET_CLASSES:
    one = kept[kept["final_label"] == label].sort_values("class_index")
    print(
        f"  {label}: {one.iloc[0]['final_filename']} -> "
        f"{one.iloc[-1]['final_filename']} ({len(one):,} images)"
    )

# Rebuild staging from source-clean crops, not from the previous flawed DATA split.
if STAGING_ROOT.exists():
    shutil.rmtree(STAGING_ROOT)
STAGING_ROOT.mkdir(parents=True, exist_ok=True)

final_rows = []

for row in kept.sort_values(["split", "final_label", "class_index"]).to_dict("records"):
    source_root = Path(row["source_dataset_root"])
    source_path = source_root / row["relative_path"]
    assert source_path.is_file(), f"Thiếu source crop: {source_path}"

    final_relative_path = f"{row['split']}/{row['final_label']}/{row['final_filename']}"
    target_path = STAGING_ROOT / final_relative_path
    target_path.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(source_path, target_path)

    out = dict(row)
    out["source_relative_path"] = out.pop("relative_path")
    out["final_relative_path"] = final_relative_path
    final_rows.append(out)

final_manifest = pd.DataFrame(final_rows)
final_manifest.to_csv(STAGING_ROOT / "manifest.csv", index=False)

readme = f"""# CS231 Final M0 Dataset — FIX2

Task: closed-set image classification of one Vietnamese polymer banknote.
Classes: {', '.join(TARGET_CLASSES)}

Source candidate pool before visual review: 9984
Classification-Centric Visual Audit V2 removals: 23
Final images: {len(final_manifest)}

Balancing:
- No class undersampling/balancing was performed.
- All samples that passed Visual Audit V2 were retained.

Split:
- Approximate target: 80% train / 10% dev / 10% test.
- Split unit is a leakage group, not an individual image.
- money_v1: same capture_date is kept in one split.
- banknotes_v2: historical capture-sequence grouping is preserved.
- exact source/crop SHA and UUID-like lineage are unioned when available.
- group integrity takes priority over exact ratio.

Naming:
- Canonical filename format: <denomination>_<6-digit-index>.png
- Index is global within each denomination and does not reset by split.
- Source identity is removed from filename but preserved in manifest.csv.

Representation:
- one-banknote source-stage logic;
- bbox short side >= 120 px on available export;
- crop = banknote bbox + 20% natural context;
- aspect ratio preserved;
- no permanent 224x224 resize.

Limitation:
- Public sources do not provide reliable physical-banknote IDs.
- Therefore the dataset does not claim physical-note-disjoint splits.
"""
(STAGING_ROOT / "README.md").write_text(readme, encoding="utf-8")

print("Staging FIX2 built:", STAGING_ROOT)
print("Manifest rows:", len(final_manifest))


Canonical filename ranges:
  10000: 10000_000001.png -> 10000_001730.png (1,730 images)
  20000: 20000_000001.png -> 20000_001475.png (1,475 images)
  50000: 50000_000001.png -> 50000_001348.png (1,348 images)
  100000: 100000_000001.png -> 100000_001671.png (1,671 images)
  200000: 200000_000001.png -> 200000_002144.png (2,144 images)
  500000: 500000_000001.png -> 500000_001593.png (1,593 images)
Staging FIX2 built: C:\CS231\DATA_NEW_M0_FIX2
Manifest rows: 9961


## 7. Final audit FIX2 trước khi thay DATA

Hard gates:

- đúng 9.961 manifest rows / PNG;
- không missing/unreferenced/unreadable;
- label ↔ path đúng;
- filename đúng canonical scheme và không chứa source prefix;
- class index liên tục 1..N trong từng class;
- `final_group` overlap train/dev/test = 0;
- `money_v1 capture_date` overlap = 0;
- `banknotes_v2 base_group_key` overlap = 0;
- exact source/crop SHA cross-split duplicate = 0 khi hash có sẵn.

Chỉ `FINAL_DATA_STATUS = PASS` mới được chạy cell thay `C:\CS231\DATA`.


In [7]:
manifest_path = STAGING_ROOT / "manifest.csv"
audit_manifest = pd.read_csv(manifest_path, keep_default_na=False, dtype=str)

manifest_rows = len(audit_manifest)
png_paths = sorted(STAGING_ROOT.rglob("*.png"))
png_count = len(png_paths)

expected_rel = set(audit_manifest["final_relative_path"].tolist())
actual_rel = {p.relative_to(STAGING_ROOT).as_posix() for p in png_paths}

missing_files = sorted(expected_rel - actual_rel)
unreferenced_png = sorted(actual_rel - expected_rel)
duplicate_relative_paths = int(audit_manifest["final_relative_path"].duplicated().sum())
duplicate_filenames = int(audit_manifest["final_filename"].duplicated().sum())
invalid_labels = int((~audit_manifest["final_label"].isin(TARGET_CLASSES)).sum())

label_path_mismatch = 0
naming_mismatch = 0
source_prefix_leak = 0
source_prefix_pattern = re.compile(r"BANKNOTES|MONEY|BLINDTRIX|CURRENCY|VR\d+", re.I)

for row in audit_manifest.to_dict("records"):
    parts = Path(row["final_relative_path"]).parts
    if len(parts) < 3 or parts[0] != row["split"] or parts[1] != row["final_label"]:
        label_path_mismatch += 1

    expected_name = f"{row['final_label']}_{int(row['class_index']):06d}.png"
    if row["final_filename"] != expected_name or Path(row["final_relative_path"]).name != expected_name:
        naming_mismatch += 1

    if source_prefix_pattern.search(row["final_filename"]):
        source_prefix_leak += 1

# Class index phải liên tục toàn class, không reset theo split.
class_index_gap = 0
for label in TARGET_CLASSES:
    one = audit_manifest[audit_manifest["final_label"] == label].copy()
    indices = sorted(int(x) for x in one["class_index"])
    if indices != list(range(1, len(one) + 1)):
        class_index_gap += 1

unreadable_png = 0
for p in png_paths:
    try:
        with Image.open(p) as image:
            image.verify()
    except Exception:
        unreadable_png += 1

# Group leakage.
group_split_n = audit_manifest.groupby("final_group")["split"].nunique()
cross_split_group_overlap = int((group_split_n > 1).sum())

money = audit_manifest[
    (audit_manifest["dataset_id"] == "money_v1")
    & (audit_manifest["capture_date"].astype(str).str.strip() != "")
]
money_date_overlap = 0
if len(money):
    money_date_overlap = int(
        (money.groupby("capture_date")["split"].nunique() > 1).sum()
    )

banknotes = audit_manifest[audit_manifest["dataset_id"] == "banknotes_v2"]
banknotes_capture_overlap = int(
    (banknotes.groupby("base_group_key")["split"].nunique() > 1).sum()
)


def cross_split_hash_overlap(column):
    if column not in audit_manifest.columns:
        return 0
    temp = audit_manifest[audit_manifest[column].astype(str).str.len() > 0]
    if len(temp) == 0:
        return 0
    counts = temp.groupby(column)["split"].nunique()
    return int((counts > 1).sum())


cross_split_source_sha = cross_split_hash_overlap("source_image_sha256")
cross_split_crop_sha = cross_split_hash_overlap("final_crop_sha256")

# Split sanity; exact 80/10/10 không bắt buộc vì group integrity ưu tiên.
split_totals = audit_manifest.groupby("split").size().reindex(["train", "dev", "test"], fill_value=0)
split_pct = split_totals / len(audit_manifest)
split_sanity = (
    0.70 <= split_pct["train"] <= 0.90
    and 0.04 <= split_pct["dev"] <= 0.20
    and 0.04 <= split_pct["test"] <= 0.20
)

checks = {
    "MANIFEST_ROWS_9961": manifest_rows == 9961,
    "PNG_FILES_9961": png_count == 9961,
    "MISSING_FILES_0": len(missing_files) == 0,
    "UNREFERENCED_PNG_0": len(unreferenced_png) == 0,
    "DUPLICATE_RELATIVE_PATH_0": duplicate_relative_paths == 0,
    "DUPLICATE_FINAL_FILENAME_0": duplicate_filenames == 0,
    "INVALID_LABELS_0": invalid_labels == 0,
    "LABEL_PATH_MISMATCH_0": label_path_mismatch == 0,
    "NAMING_MISMATCH_0": naming_mismatch == 0,
    "SOURCE_PREFIX_IN_FILENAME_0": source_prefix_leak == 0,
    "CLASS_INDEX_GAP_0": class_index_gap == 0,
    "UNREADABLE_PNG_0": unreadable_png == 0,
    "CROSS_SPLIT_GROUP_OVERLAP_0": cross_split_group_overlap == 0,
    "MONEY_CAPTURE_DATE_OVERLAP_0": money_date_overlap == 0,
    "BANKNOTES_CAPTURE_GROUP_OVERLAP_0": banknotes_capture_overlap == 0,
    "CROSS_SPLIT_SOURCE_SHA_0": cross_split_source_sha == 0,
    "CROSS_SPLIT_CROP_SHA_0": cross_split_crop_sha == 0,
    "SPLIT_SANITY": bool(split_sanity),
}

FINAL_DATA_STATUS = "PASS" if all(checks.values()) else "HOLD"

display(pd.DataFrame([{"check": k, "pass": v} for k, v in checks.items()]))

print("\nFINAL_DATA_STATUS =", FINAL_DATA_STATUS)
print("Manifest rows:", manifest_rows)
print("PNG files:", png_count)
print("Final leakage groups:", audit_manifest["final_group"].nunique())
print("Largest group size:", int(audit_manifest.groupby("final_group").size().max()))
print("Cross-split group overlap:", cross_split_group_overlap)
print("money_v1 capture_date overlap:", money_date_overlap)
print("banknotes_v2 capture-group overlap:", banknotes_capture_overlap)
print("Cross-split source SHA duplicates:", cross_split_source_sha)
print("Cross-split crop SHA duplicates:", cross_split_crop_sha)

print("\nFinal class totals:")
display(
    audit_manifest.groupby("final_label")
    .size()
    .reindex(TARGET_CLASSES, fill_value=0)
    .to_frame("images")
)

print("\nFinal split x class:")
display(
    audit_manifest.groupby(["split", "final_label"])
    .size()
    .unstack(fill_value=0)
    .reindex(index=["train", "dev", "test"], columns=TARGET_CLASSES, fill_value=0)
)

print("\nFinal split percentages:")
display((split_pct * 100).round(2).to_frame("percent"))

print("\nCanonical filename ranges:")
for label in TARGET_CLASSES:
    one = audit_manifest[audit_manifest["final_label"] == label].copy()
    one["class_index_num"] = one["class_index"].astype(int)
    one = one.sort_values("class_index_num")
    print(f"  {label}: {one.iloc[0]['final_filename']} -> {one.iloc[-1]['final_filename']}")

assert FINAL_DATA_STATUS == "PASS", (
    "Final FIX2 staging audit HOLD. KHÔNG chạy cell thay DATA bên dưới."
)


,check,pass
0,MANIFEST_ROWS_9961,True
1,PNG_FILES_9961,True
2,MISSING_FILES_0,True
3,UNREFERENCED_PNG_0,True
4,DUPLICATE_RELATIVE_PATH_0,True
5,DUPLICATE_FINAL_FILENAME_0,True
6,INVALID_LABELS_0,True
7,LABEL_PATH_MISMATCH_0,True
8,NAMING_MISMATCH_0,True
9,SOURCE_PREFIX_IN_FILENAME_0,True



FINAL_DATA_STATUS = PASS
Manifest rows: 9961
PNG files: 9961
Final leakage groups: 2446
Largest group size: 1718
Cross-split group overlap: 0
money_v1 capture_date overlap: 0
banknotes_v2 capture-group overlap: 0
Cross-split source SHA duplicates: 0
Cross-split crop SHA duplicates: 0

Final class totals:


,images
final_label,
10000,1730
20000,1475
50000,1348
100000,1671
200000,2144
500000,1593



Final split x class:


final_label,10000,20000,50000,100000,200000,500000
split,,,,,,
train,1384,1180,1078,1337,1715,1275
dev,173,147,135,167,215,159
test,173,148,135,167,214,159



Final split percentages:


,percent
split,
train,80.0
dev,10.0
test,10.0



Canonical filename ranges:
  10000: 10000_000001.png -> 10000_001730.png
  20000: 20000_000001.png -> 20000_001475.png
  50000: 50000_000001.png -> 50000_001348.png
  100000: 100000_000001.png -> 100000_001671.png
  200000: 200000_000001.png -> 200000_002144.png
  500000: 500000_000001.png -> 500000_001593.png


## 8. Thay `C:\CS231\DATA` hiện tại bằng FIX2 — chỉ chạy sau PASS

Bản DATA hiện tại là output FIX1. FIX2 sẽ:

1. backup DATA hiện tại;
2. move `DATA_NEW_M0_FIX2` thành `DATA`;
3. post-move audit trực tiếp trên canonical DATA;
4. chỉ khi PASS mới xóa backup;
5. nếu lỗi thì rollback.

Cell mặc định **không thay gì**. Chỉ đổi `DO_REPLACE_DATA = True` sau khi thấy `FINAL_DATA_STATUS = PASS`.


In [10]:
DO_REPLACE_DATA = True  # <-- Đổi True CHỈ SAU KHI FINAL_DATA_STATUS = PASS.

CANONICAL_DATA_STATUS = "NOT_RUN"

if not DO_REPLACE_DATA:
    print("SAFE STOP: chưa thay DATA bằng FIX2.")
    print("Khi FINAL_DATA_STATUS = PASS, đổi DO_REPLACE_DATA = True rồi chạy lại cell này.")
else:
    assert FINAL_DATA_STATUS == "PASS"
    assert STAGING_ROOT.is_dir(), f"Không tìm thấy staging FIX2: {STAGING_ROOT}"

    stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    backup = PROJECT_ROOT / f"DATA_FIX1_PENDING_DELETE_{stamp}"
    failed_new = PROJECT_ROOT / f"DATA_FIX2_FAILED_{stamp}"

    assert not backup.exists()
    assert not failed_new.exists()

    old_existed = FINAL_DATA_ROOT.exists()

    try:
        if old_existed:
            FINAL_DATA_ROOT.rename(backup)
            print("Đã backup DATA hiện tại:", backup)

        STAGING_ROOT.rename(FINAL_DATA_ROOT)
        print("Đã đưa FIX2 vào canonical path:", FINAL_DATA_ROOT)

        post_manifest_path = FINAL_DATA_ROOT / "manifest.csv"
        assert post_manifest_path.is_file()
        post = pd.read_csv(post_manifest_path, keep_default_na=False, dtype=str)
        post_png = list(FINAL_DATA_ROOT.rglob("*.png"))

        assert len(post) == 9961
        assert len(post_png) == 9961
        assert post["final_relative_path"].duplicated().sum() == 0
        assert post["final_filename"].duplicated().sum() == 0
        assert set(post["final_label"]).issubset(TARGET_CLASSES)
        assert int((post.groupby("final_group")["split"].nunique() > 1).sum()) == 0

        post_expected = set(post["final_relative_path"])
        post_actual = {p.relative_to(FINAL_DATA_ROOT).as_posix() for p in post_png}
        assert post_expected == post_actual

        # Re-check money date leakage after move.
        post_money = post[
            (post["dataset_id"] == "money_v1")
            & (post["capture_date"].astype(str).str.strip() != "")
        ]
        if len(post_money):
            assert int((post_money.groupby("capture_date")["split"].nunique() > 1).sum()) == 0

        CANONICAL_DATA_STATUS = "PASS"

        if old_existed and backup.exists():
            shutil.rmtree(backup)
            print("Đã xóa DATA FIX1 cũ sau post-move PASS.")

        print("\nCANONICAL_DATA_STATUS = PASS")
        print("Canonical DATA FIX2:", FINAL_DATA_ROOT)

    except Exception:
        CANONICAL_DATA_STATUS = "HOLD"
        print("\nPOST-MOVE CHECK FAILED -> rollback...")

        if FINAL_DATA_ROOT.exists():
            FINAL_DATA_ROOT.rename(failed_new)

        if old_existed and backup.exists():
            backup.rename(FINAL_DATA_ROOT)
            print("Đã phục hồi DATA trước FIX2:", FINAL_DATA_ROOT)

        raise


Đã backup DATA hiện tại: C:\CS231\DATA_FIX1_PENDING_DELETE_20261026_121610
Đã đưa FIX2 vào canonical path: C:\CS231\DATA
Đã xóa DATA FIX1 cũ sau post-move PASS.

CANONICAL_DATA_STATUS = PASS
Canonical DATA FIX2: C:\CS231\DATA


## 9. Kiểm tra notebook M0 legacy

Bạn đã xóa 5 notebook cũ ở lần chạy trước. FIX2 chỉ **verify** trạng thái đó; nếu vì lý do nào đó legacy notebook vẫn còn, có thể bật cờ cleanup sau khi canonical DATA FIX2 PASS.

Notebook 05 FIX2 nên được lưu/đổi tên thành `05_BUILD_FINAL_DATASET_SIMPLE.ipynb` trong `src\M0` để M0 chỉ có đúng bộ SIMPLE 01–05.


In [11]:
DELETE_LEGACY_NOTEBOOKS = True

NEW_CANONICAL_NOTEBOOKS = [
    "01_BANKNOTES_RAW_SIMPLE.ipynb",
    "02_MONEY_RAW_SIMPLE.ipynb",
    "03_MONEY_VN_RAW_SIMPLE.ipynb",
    "04_VIETNAMESE_CURRENCY_RAW_SIMPLE.ipynb",
    "05_BUILD_FINAL_DATASET_SIMPLE.ipynb",
]

LEGACY_NOTEBOOKS = [
    "01_BANKNOTES_RAW_TO_377.ipynb",
    "02_MONEY_RAW.ipynb",
    "03_MONEY_VN_RAW.ipynb",
    "04_VIETNAMESE_CURRENCY_RAW.ipynb",
    "05_BUILD_FINAL_DATASET.ipynb",
]

legacy_present = [name for name in LEGACY_NOTEBOOKS if (M0_DIR / name).is_file()]
print("Legacy notebooks còn hiện diện:", legacy_present if legacy_present else "NONE")

if DELETE_LEGACY_NOTEBOOKS:
    assert CANONICAL_DATA_STATUS == "PASS", (
        "Chỉ cleanup notebook sau canonical DATA FIX2 PASS."
    )
    for name in legacy_present:
        (M0_DIR / name).unlink()
        print("Deleted:", name)

remaining = sorted(p.name for p in M0_DIR.glob("*.ipynb"))
print("\nCurrent .ipynb files in src\\M0:")
for name in remaining:
    print("  -", name)

remaining_legacy = [name for name in LEGACY_NOTEBOOKS if name in remaining]
M0_NOTEBOOK_CLEANUP_STATUS = "PASS" if not remaining_legacy else "HOLD"
print("\nM0_NOTEBOOK_CLEANUP_STATUS =", M0_NOTEBOOK_CLEANUP_STATUS)


Legacy notebooks còn hiện diện: NONE

Current .ipynb files in src\M0:
  - 01_BANKNOTES_RAW_SIMPLE.ipynb
  - 02_MONEY_RAW_SIMPLE.ipynb
  - 03_MONEY_VN_RAW_SIMPLE.ipynb
  - 04_VIETNAMESE_CURRENCY_RAW_SIMPLE.ipynb
  - 05_BUILD_FINAL_DATASET_SIMPLE.ipynb

M0_NOTEBOOK_CLEANUP_STATUS = PASS


## Kết thúc M0 — FIX2

M0 chỉ được freeze khi có:

```text
FINAL_DATA_STATUS = PASS
CANONICAL_DATA_STATUS = PASS
M0_NOTEBOOK_CLEANUP_STATUS = PASS
```

Khi đó canonical state:

- `C:\CS231\DATA` = **9.961 ảnh**;
- không balance;
- split theo real provenance group, không phải image-level;
- `money_v1` cùng capture date không xuyên split;
- `banknotes_v2` cùng capture lineage không xuyên split;
- filename thống nhất `<denomination>_<6-digit-index>.png`;
- provenance vẫn đầy đủ trong `manifest.csv`;
- M0 chỉ còn notebook SIMPLE 01–05.

Sau FIX2 PASS mới chuyển sang M1/modeling.
